# Model B — YOLOv8s one-stage detector
**Owner:** Dhruv Maheshwari

YOLOv8s detects and localises damage on full images. 30 epochs at 640 px ≈ 3–3.5 h on a T4 (reduce `--epochs` to 20 if short of time; early stopping is on).

**Before running:** Kaggle → *Settings* → Accelerator **GPU T4 x2**, Internet **On**. *Add Input* → dataset **aliabdelmenam/rdd-2022** (the same RDD2022 `RDD_SPLIT` used in Phase 2). Then *Run All*.
The final cell zips `results/` → download it from the Output panel and put it into the repo's `results/` folder.

In [ ]:
import os, subprocess, sys
# Option A (recommended): the final code is pushed to the team repo
if not os.path.exists('repo'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/lakshya1333/Road-damage-detection-deep-learning-project', 'repo'], check=True)
# Option B: if you uploaded the code folder as a Kaggle dataset instead, copy it:  !cp -r /kaggle/input/<your-code-dataset> repo
os.chdir('repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.2', 'segmentation-models-pytorch>=0.3'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Shared manifest: identical split + class mapping for all four models (check the printed class mapping!)
!python -m src.data.build_manifest --out results/data --plot

In [ ]:
!python -m src.models.train_yolo --manifest results/data --out results/yolov8 --model yolov8s.pt --epochs 30 --imgsz 640 --batch 16

In [ ]:
# Optional: video monitoring demo on any dashcam clip uploaded as input
# !python -m src.inference.video_infer --weights results/yolov8/train/weights/best.pt --video /kaggle/input/<clip>/road.mp4 --fps 5

In [ ]:
from IPython.display import Image, display
display(Image('results/yolov8/train/results.png'))
display(Image('results/yolov8/test/confusion_matrix_normalized.png'))

In [ ]:
import shutil, glob, os
# drop large weights from the archive except the best checkpoint
shutil.make_archive('/kaggle/working/results_yolov8', 'zip', 'results')
print(open(sorted(glob.glob('results/*/metrics.json'))[0]).read()[:3000])